<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/model-risk/lessons/P04-L08-explainability-evidence/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/model-risk/lessons/P04-L08-explainability-evidence/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/model-risk/lessons/P04-L08-explainability-evidence/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/model-risk/lessons/P04-L08-explainability-evidence/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P04-L08 · Explainability evidence that reproduces

**You will build:** permutation importance with a repeat count and a spread, the grouped
version that fixes what correlated features do to it, one- and two-dimensional partial
dependence, a measure of how far partial dependence strays from the data, exact Shapley
attributions by enumerating every coalition — and a reproducibility check that regenerates
the whole explanation pack in fresh processes and compares it byte for byte.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** T00-L01 (the tier gate and the profiler) and P04-L01 (the validation
suite, its house conventions, and `auc_by_ranks`). Pure numpy and the standard library:
there is no scikit-learn here and no SHAP package.

The data is **synthetic and generated in this notebook**. Every figure you see is computed
by code you run.

By the end you will be able to:

1. Implement permutation importance on held-out data with a repeat count and a reported
   spread, and measure why two strongly correlated features each receive little of the
   importance their pair carries — and fix it by permuting the pair together.
2. Implement one- and two-dimensional partial dependence, and measure how many of the points
   it averages over lie outside the region the data actually occupies.
3. Implement exact Shapley attributions under a stated baseline value function, verify that
   they sum to the prediction minus the baseline, and measure the exponential cost.
4. Implement a canonical serialisation and a reproducibility check that regenerates the pack
   in fresh processes and asserts byte equality.
5. Explain why a check that regenerates in the same process certifies a pack whose numbers
   depend on the process's hash seed.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import builtins
import contextlib
import dis
import hashlib
import io
import json
import marshal
import math
import os
import pickle
import platform
import subprocess
import sys
import time
import traceback
import types
from typing import Any, Callable

import numpy as np

SEED = 20260923
N_DEV = 4000            # development sample: what the champion was fitted on
N_HOLD = 2000           # held-out sample: what every explanation is computed on
RHO_INCOME = 0.95       # population correlation between declared and bureau income
FEATURES = ("declared_income", "bureau_income", "utilisation", "missed_payments",
            "file_age", "loan_to_income")
N_REPEATS = 10          # permutations per feature: the spread comes from these
IMPORTANCE_GROUPS = ((0, 1), (2,), (3,), (4,), (5,))   # the income pair, then the rest alone
GRID = tuple(-2.5 + 0.5 * i for i in range(11))        # standardised units, both ends included
GRID_2D = tuple(-2.0 + 0.5 * i for i in range(9))
PD_FEATURES = ("declared_income", "bureau_income", "utilisation")
MANIFOLD_LEVEL = 0.99   # the ellipse that holds this share of a bivariate normal
MANIFOLD_PAIRS = ((0, 1), (2, 0))   # (feature, partner): the income pair, and a control
N_BACKGROUND = 64       # development rows that stand in for "an applicant we know nothing about"
HASH_SEEDS = (1, 2, 3)  # PYTHONHASHSEED for each fresh process the check starts
# The model risk committee's request, as it arrived: it names utilisation twice.
COMMITTEE_ASKED = ("declared_income", "bureau_income", "utilisation", "missed_payments",
                   "file_age", "loan_to_income", "utilisation")

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__)

DATA_NOTE = (
    "SYNTHETIC DATA. Every applicant in this notebook was generated inside it by "
    f"numpy.random.default_rng({SEED}). No real applicant, account or lending decision is "
    "represented. Declared income and bureau income are two noisy readings of one latent "
    "income, built to be strongly correlated on purpose."
)
MODEL_NOTE = ("logistic regression on the six features, fitted by Newton's method with a "
              "unit ridge penalty; coefficients frozen at 6 decimal places")

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("permutation_importance",),
    "exercise 2": ("grouped_permutation_importance",),
    "exercise 3": ("partial_dependence",),
    "exercise 4": ("partial_dependence_2d",),
    "exercise 5": ("off_manifold_share",),
    "exercise 6": ("coalition_value",),
    "exercise 7": ("shapley_values",),
    "exercise 8": ("canonical_bytes",),
    "exercise 9": ("reproducibility_check",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (partial_dependence)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def auc_by_ranks(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Area under the ROC curve by the Mann-Whitney rank identity. Given to you, not graded.

    A minimal faithful copy of the function P04-L01 handed you and P04-L04 made you build:
    the same name, the same arguments, ties given the average rank, and a ValueError when
    one class is absent. Higher is better, which is what permutation importance needs.
    """
    y = np.asarray(y_true)
    s = np.asarray(y_score, dtype=float)
    n_pos = int(y.sum())
    n_neg = int(y.size - n_pos)
    if n_pos == 0 or n_neg == 0:
        raise ValueError("AUC is undefined when one class is absent from the sample")
    order = np.argsort(s, kind="mergesort")
    _, inverse, counts = np.unique(s[order], return_inverse=True, return_counts=True)
    ends = np.cumsum(counts)
    average_rank = (ends - counts + 1 + ends) / 2.0     # mean of ranks start+1 .. end
    ranks = np.empty(s.size)
    ranks[order] = average_rank[inverse]
    return float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def make_book(seed: int) -> dict:
    """Generate the development and held-out samples from `seed`. SYNTHETIC — see DATA_NOTE.

    Given to you. Declared income and bureau income are the same latent income read twice,
    each with its own noise, scaled so both have unit variance and their correlation is
    RHO_INCOME. Every other feature is independent. The true risk depends on the LATENT
    income, not on either reading — so any model has to split the credit between the two.
    """
    rng = np.random.default_rng(seed)
    n = N_DEV + N_HOLD
    income = rng.normal(0.0, 1.0, n)
    spread = math.sqrt(1.0 / RHO_INCOME - 1.0)
    scale = math.sqrt(RHO_INCOME)
    declared = scale * (income + spread * rng.normal(0.0, 1.0, n))
    bureau = scale * (income + spread * rng.normal(0.0, 1.0, n))
    utilisation = rng.normal(0.0, 1.0, n)
    missed = rng.normal(0.0, 1.0, n)
    file_age = rng.normal(0.0, 1.0, n)
    loan_to_income = rng.normal(0.0, 1.0, n)
    true_logit = (-2.2 - 0.9 * income + 0.75 * utilisation + 0.6 * missed
                  - 0.35 * file_age + 0.45 * loan_to_income)
    y = (rng.random(n) < 1.0 / (1.0 + np.exp(-true_logit))).astype(np.int64)
    X = np.column_stack([declared, bureau, utilisation, missed, file_age, loan_to_income])
    return {"X_dev": X[:N_DEV], "y_dev": y[:N_DEV], "X_hold": X[N_DEV:], "y_hold": y[N_DEV:]}


def fit_logistic(X: np.ndarray, y: np.ndarray, ridge: float = 1.0) -> np.ndarray:
    """Logistic regression by Newton's method, intercept first. Given to you, not graded.

    Module 7 of this programme builds this properly. Here it only has to produce the model
    under review. The unit ridge penalty is what makes a model split its weight evenly
    between two near-copies of one signal, which is the realistic case. The coefficients are
    frozen at 6 decimal places, the way a model inventory records a scorecard, so the model
    under review is a fixed object rather than whatever the last bit of a solver produced.
    """
    A = np.column_stack([np.ones(X.shape[0]), X])
    penalty = np.eye(A.shape[1]) * ridge
    penalty[0, 0] = 0.0
    w = np.zeros(A.shape[1])
    for _ in range(50):
        p = 1.0 / (1.0 + np.exp(-(A @ w)))
        gradient = A.T @ (p - y) + penalty @ w
        hessian = (A * (p * (1.0 - p))[:, None]).T @ A + penalty
        step = np.linalg.solve(hessian, gradient)
        w = w - step
        if np.max(np.abs(step)) < 1e-10:
            break
    return np.round(w, 6)


def champion_logit(coef: np.ndarray, X: np.ndarray) -> np.ndarray:
    """The model's score on the log-odds scale. Given to you.

    One column at a time rather than a matrix product: an elementwise multiply-and-add gives
    the same bits on every machine, where a BLAS kernel is free to reorder a sum.
    """
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or X.shape[1] != len(coef) - 1:
        raise ValueError(f"expected a 2-D array with {len(coef) - 1} columns")
    z = np.full(X.shape[0], float(coef[0]))
    for j in range(X.shape[1]):
        z = z + float(coef[j + 1]) * X[:, j]
    return z


def champion_probability(coef: np.ndarray, X: np.ndarray) -> np.ndarray:
    """The model's predicted probability of default. Given to you."""
    return 1.0 / (1.0 + np.exp(-champion_logit(coef, X)))


BOOK = make_book(SEED)
COEF = fit_logistic(BOOK["X_dev"], BOOK["y_dev"])
X_HOLD, Y_HOLD = BOOK["X_hold"], BOOK["y_hold"]


def champion_score(X: np.ndarray) -> np.ndarray:
    """The frozen champion's log-odds. AUC only needs the ranking, so this is what it scores."""
    return champion_logit(COEF, X)


def champion_prob(X: np.ndarray) -> np.ndarray:
    """The frozen champion's probability of default: what partial dependence and Shapley use."""
    return champion_probability(COEF, X)


print(f"development sample: {N_DEV} applicants, default rate {BOOK['y_dev'].mean():.4f}")
print(f"held-out sample:    {N_HOLD} applicants, default rate {Y_HOLD.mean():.4f}")
print("\n" + DATA_NOTE)

## 1. Why an explanation is evidence, and the model under review

Under the EU AI Act, AI systems intended to evaluate the creditworthiness of natural persons
or establish their credit score are high-risk systems listed in Annex III (fraud detection
excepted), and after the Digital Omnibus the high-risk requirements and obligations in
Sections 1 to 3 of Chapter III apply to Annex III systems from 2 December 2027. Article 86
gives a person affected by a decision taken on the basis of such a system's output the right
to clear and meaningful explanations of the role the system played. An explanation handed to
an applicant, a supervisor or a court is evidence. **One that cannot be regenerated from the
run that produced it is an anecdote.**

The model under review is a frozen logistic regression on six standardised features. Two of
them — the income the applicant declared and the income the credit bureau reports — are
two readings of one thing, and that is where every explanation method in this notebook is
going to be tested. Run the cell: it prints how tightly the two readings move together and
how the fitted model split its weight between them.

In [ ]:
_r = float(np.corrcoef(X_HOLD[:, 0], X_HOLD[:, 1])[0, 1])
print(f"correlation of declared and bureau income, held-out sample: {_r:.3f}")
print("frozen champion coefficients (log-odds per standard deviation):")
print(f"  {'intercept':<16} {COEF[0]:+.6f}")
for _j, _name in enumerate(FEATURES):
    print(f"  {_name:<16} {COEF[_j + 1]:+.6f}")
print(f"held-out AUC: {auc_by_ranks(Y_HOLD, champion_score(X_HOLD)):.4f}")
print("\nThe two income readings carry one effect between them. Keep an eye on what each")
print("explanation method below does with that.")

## 2. Exercise 1 — `permutation_importance()`

Breiman introduced the idea with random forests: randomly permute one variable's values in
the out-of-bag rows a tree had not been fitted on — which breaks its link to the outcome
while keeping its distribution — and measure how much worse the model gets. Here "worse"
is the fall in held-out AUC.

A single shuffle is one draw from a random procedure, so it is not a measurement until you
repeat it and say how much it moved. That is why this function takes a repeat count and a
random generator, and returns a spread beside every mean. It also takes the ORDER in which
to score the columns — and uses one generator across all of them. Remember that; section 12
is about it.

<details><summary>💡 Hint 1 — what to think about</summary>

Three things carry the marks. Every model call after the baseline changes exactly one column
and leaves every other column — and your caller's array — exactly as they were. Each repeat
needs its own fresh shuffle, or the spread you report is a spread of one number. And the
generator you were handed is the only source of randomness: the same seed must give the same
answer, bit for bit.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate the shapes, the repeat count and the column indices first. Score the untouched data
once for the baseline. Then walk the columns in the order given; for each one and each
repeat, draw a permutation of the row indices from the generator, copy the data, replace that
one column with its values in permuted row order, and record the baseline minus the new
score. Report the mean over repeats and the sample standard deviation across them.

</details>

In [ ]:
def permutation_importance(model: Callable[[np.ndarray], np.ndarray],
                           metric: Callable[[np.ndarray, np.ndarray], float],
                           X: np.ndarray, y: np.ndarray, n_repeats: int,
                           rng: np.random.Generator, columns=None) -> dict:
    """Permutation importance of each column in `columns`, with a spread over repeats.

    `model(X)` returns one score per row; `metric(y, scores)` returns a float where HIGHER is
    better (AUC, here). Pass held-out data: an importance measured on the rows the model was
    fitted on measures what it memorised as well as what it learned.

    Requirements, each graded:
      * `baseline` is `metric(y, model(X))` on the untouched data, as a Python float.
      * for each column j in `columns` (default: every column, left to right) and each of the
        `n_repeats` repeats, in that order: draw `rng.permutation(n_rows)`, copy X, replace
        column j with its values in that permuted row order, and record
        `baseline - metric(y, model(copy))`. Only column j changes; every other column, and
        the caller's X, stay exactly as they were.
      * `drops` has shape `(len(columns), n_repeats)`; `mean` and `std` are taken across the
        repeats, `std` as the SAMPLE standard deviation (ddof=1). A spread from one repeat is
        not a spread, so `n_repeats < 2` is a ValueError.
      * all randomness comes from `rng`: the same seed gives the same result, bit for bit.
      * `ValueError` if X is not 2-D, if y does not match its rows, or a column is out of range.

    Example:
        >>> X = np.array([[0.0, 5.0], [1.0, 5.0], [2.0, 5.0], [3.0, 5.0]])
        >>> y = np.array([0, 0, 1, 1])
        >>> r = permutation_importance(lambda A: A[:, 0], auc_by_ranks, X, y, 3,
        ...                            np.random.default_rng(0))
        >>> r["baseline"], r["columns"], r["drops"].shape
        (1.0, (0, 1), (2, 3))

    Returns:
        dict with `baseline` (float), `columns` (tuple of int, in scoring order), `mean` and
        `std` (float arrays, one entry per column) and `drops` (the raw 2-D array).
    """
    # YOUR CODE HERE
    raise NotImplementedError


class _Spy:
    """A model that records every matrix it is shown. Used by the checks, not by you."""

    def __init__(self, fn: Callable[[np.ndarray], np.ndarray]):
        self.fn, self.seen = fn, []

    def __call__(self, X: np.ndarray) -> np.ndarray:
        self.seen.append(np.array(X, dtype=float, copy=True))
        return self.fn(X)


def _neg_mse(y: np.ndarray, s: np.ndarray) -> float:
    return -float(np.mean((np.asarray(y, dtype=float) - s) ** 2))


def _check_permutation_importance() -> None:
    rng = np.random.default_rng(7)
    X = rng.normal(0.0, 1.0, (30, 3))
    y = X[:, 0] - 0.5 * X[:, 2]
    before = X.copy()
    spy = _Spy(lambda A: A[:, 0] - 0.5 * A[:, 2])
    res = permutation_importance(spy, _neg_mse, X, y, 4, np.random.default_rng(1))
    assert np.array_equal(X, before), (
        "your function changed the caller's X — shuffle a COPY, never the array you were given"
    )
    assert isinstance(res, dict) and {"baseline", "columns", "mean", "std", "drops"} <= set(res), (
        "return a dict with baseline, columns, mean, std and drops"
    )
    assert np.asarray(res["drops"]).shape == (3, 4), (
        f"drops has shape {np.asarray(res['drops']).shape}; expected (columns, repeats) = (3, 4)"
    )
    changed = []
    for A in spy.seen:
        cols = [j for j in range(3) if not np.array_equal(A[:, j], X[:, j])]
        assert len(cols) <= 1, (
            f"one model call changed columns {cols} at once — only the column being scored may "
            "move; if earlier columns are still shuffled you permuted in place without copying"
        )
        if cols:
            j = cols[0]
            assert np.array_equal(np.sort(A[:, j]), np.sort(X[:, j])), (
                "a shuffled column must hold the same values in a different row order"
            )
            changed.append(j)
    assert [changed.count(j) for j in range(3)] == [4, 4, 4], (
        f"each column should be shuffled once per repeat; counts were "
        f"{[changed.count(j) for j in range(3)]}"
    )
    base = _neg_mse(y, X[:, 0] - 0.5 * X[:, 2])
    assert abs(res["baseline"] - base) < 1e-12, "baseline is the metric on the untouched data"
    assert abs(res["mean"][1]) < 1e-12, (
        "column 1 is never used by the model, so its importance must be exactly zero"
    )
    assert res["mean"][0] > 0, (
        "shuffling a feature the model relies on must LOWER the metric, so the drop "
        "(baseline minus shuffled) is positive — check the order of the subtraction"
    )
    d = np.asarray(res["drops"])
    assert np.allclose(res["std"], d.std(axis=1, ddof=1)), (
        "std is the SAMPLE standard deviation across repeats (ddof=1), not the population one"
    )
    again = permutation_importance(lambda A: A[:, 0] - 0.5 * A[:, 2], _neg_mse, X, y, 4,
                                   np.random.default_rng(1))
    assert np.array_equal(again["drops"], d), (
        "the same seed gave different drops — draw every permutation from the rng you were "
        "handed, never from np.random or a fresh unseeded generator"
    )
    rev = permutation_importance(lambda A: A[:, 0] - 0.5 * A[:, 2], _neg_mse, X, y, 4,
                                 np.random.default_rng(1), columns=(1, 0))
    assert tuple(rev["columns"]) == (1, 0) and abs(rev["mean"][0]) < 1e-12 \
        and rev["mean"][1] > 0, (
        "with columns=(1, 0), entry 0 of mean belongs to column 1 (unused, so zero) and "
        "entry 1 to column 0 — score the columns in the order you were given"
    )
    try:
        permutation_importance(lambda A: A[:, 0], _neg_mse, X, y, 1, np.random.default_rng(1))
    except ValueError:
        pass
    else:
        raise AssertionError("n_repeats=1 must raise ValueError: one repeat has no spread")
    print("exercise 1 looks right")

In [ ]:
_try("exercise 1", _check_permutation_importance)

In [ ]:
def _show_single_importance() -> None:
    res = permutation_importance(champion_score, auc_by_ranks, X_HOLD, Y_HOLD, N_REPEATS,
                                 np.random.default_rng(SEED))
    print(f"held-out AUC {res['baseline']:.4f}; importance = fall in AUC when the feature is "
          f"shuffled, {N_REPEATS} repeats")
    for a in np.argsort(-res["mean"], kind="stable"):
        name = FEATURES[res["columns"][a]]
        print(f"  {name:<16} {res['mean'][a]:.4f}  ± {res['std'][a]:.4f}")


_try("single-feature importance", _show_single_importance, needs=("exercise 1",))

## 3. Exercise 2 — `grouped_permutation_importance()`

Look at where the two income readings landed in that ranking. The generator made latent
income the strongest driver of default, and yet each reading, shuffled on its own, ranks
below features the generator made weaker. Nothing is wrong with the arithmetic. When one
reading is shuffled, the model still has the other, which carries almost the same
information — so the damage is small, and it is small for both.

The fix is to shuffle the pair together: one row permutation applied to both columns, so
every applicant keeps a realistic pair of incomes and only the pair's link to the outcome is
broken. That is grouped permutation importance.

<details><summary>💡 Hint 1 — what to think about</summary>

"Together" is the whole exercise. If the two columns of a group get two different shuffles,
you manufacture applicants whose declared and bureau incomes disagree wildly — people who do
not exist — and you are measuring something else. One permutation per group per repeat,
applied to every column in that group at once.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

The same skeleton as exercise 1, with a group of column indices where a single column was.
Validate that every group is non-empty and every index is in range. For each group and each
repeat, draw one permutation, copy the data, and overwrite all of the group's columns with
those columns' values taken in that one permuted row order.

</details>

In [ ]:
def grouped_permutation_importance(model: Callable[[np.ndarray], np.ndarray],
                                   metric: Callable[[np.ndarray, np.ndarray], float],
                                   X: np.ndarray, y: np.ndarray, groups, n_repeats: int,
                                   rng: np.random.Generator) -> dict:
    """Permutation importance of GROUPS of columns, each group shuffled jointly.

    Same contract as `permutation_importance`, with `groups` — a sequence of sequences of
    column indices — in place of `columns`. Requirements, each graded:
      * for each group, in order, and each repeat: ONE `rng.permutation(n_rows)`, applied to
        every column of the group at once, so each row keeps a real combination of the
        group's values. Columns outside the group, and the caller's X, do not change.
      * `drops[g, r]` is `baseline - metric(y, model(shuffled))`; `mean`, and `std` at
        ddof=1, are across repeats.
      * `ValueError` for an empty group, an index out of range, `n_repeats < 2`, a non-2-D X
        or a y of the wrong length.

    Example:
        >>> X = np.array([[0.0, 0.0], [1.0, 1.0], [2.0, 2.0], [3.0, 3.0]])
        >>> r = grouped_permutation_importance(lambda A: A[:, 0] + A[:, 1], auc_by_ranks, X,
        ...                                    np.array([0, 0, 1, 1]), [(0, 1)], 2,
        ...                                    np.random.default_rng(0))
        >>> r["groups"], r["drops"].shape
        (((0, 1),), (1, 2))

    Returns:
        dict with `baseline` (float), `groups` (tuple of tuples of int), `mean`, `std` (one
        entry per group) and `drops` (shape `(len(groups), n_repeats)`).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_grouped() -> None:
    rng = np.random.default_rng(11)
    X = rng.normal(0.0, 1.0, (40, 3))
    y = X[:, 0] + X[:, 1]
    before = X.copy()
    spy = _Spy(lambda A: A[:, 0] + A[:, 1])
    res = grouped_permutation_importance(spy, _neg_mse, X, y, [(0, 1), (2,)], 3,
                                         np.random.default_rng(2))
    assert np.array_equal(X, before), "your function changed the caller's X — shuffle a copy"
    assert np.asarray(res["drops"]).shape == (2, 3), (
        f"drops has shape {np.asarray(res['drops']).shape}; expected (groups, repeats) = (2, 3)"
    )
    original_rows = sorted(map(tuple, X[:, :2].tolist()))
    pair_calls = 0
    for A in spy.seen:
        if not np.array_equal(A[:, 0], X[:, 0]):
            pair_calls += 1
            assert np.array_equal(A[:, 2], X[:, 2]), (
                "shuffling the group (0, 1) must leave column 2 alone"
            )
            assert sorted(map(tuple, A[:, :2].tolist())) == original_rows, (
                "after shuffling the pair, some rows hold a combination of the two columns that "
                "no real row had — the two columns got DIFFERENT permutations; draw one "
                "permutation per group and apply it to every column in the group"
            )
    assert pair_calls == 3, f"the pair should be shuffled once per repeat, saw {pair_calls}"
    assert abs(res["mean"][1]) < 1e-12, (
        "column 2 is not used by the model, so the group (2,) must score exactly zero"
    )
    try:
        grouped_permutation_importance(lambda A: A[:, 0], _neg_mse, X, y, [()], 3,
                                       np.random.default_rng(2))
    except ValueError:
        pass
    else:
        raise AssertionError("an empty group must raise ValueError")
    print("exercise 2 looks right")

In [ ]:
_try("exercise 2", _check_grouped)

Now measure it. The cell below scores each income reading on its own, then the pair shuffled
together, on the same held-out rows with the same number of repeats — and ranks the features
both ways, which is the ranking a committee would actually read.

In [ ]:
def _show_grouped_versus_single() -> None:
    single = permutation_importance(champion_score, auc_by_ranks, X_HOLD, Y_HOLD, N_REPEATS,
                                    np.random.default_rng(SEED))
    grouped = grouped_permutation_importance(champion_score, auc_by_ranks, X_HOLD, Y_HOLD,
                                             IMPORTANCE_GROUPS, N_REPEATS,
                                             np.random.default_rng(SEED))
    s_dec, s_bur = float(single["mean"][0]), float(single["mean"][1])
    pair = float(grouped["mean"][0])
    print(f"declared income, shuffled alone:  {s_dec:.4f} ± {single['std'][0]:.4f}")
    print(f"bureau income, shuffled alone:    {s_bur:.4f} ± {single['std'][1]:.4f}")
    print(f"sum of the two single figures:    {s_dec + s_bur:.4f}")
    print(f"the pair, shuffled together:      {pair:.4f} ± {grouped['std'][0]:.4f}")
    print(f"the pair carries {pair / (s_dec + s_bur):.2f} times the sum of its parts\n")
    by_single = [FEATURES[single["columns"][a]] for a in np.argsort(-single["mean"],
                                                                     kind="stable")]
    labels = ["+".join(FEATURES[j] for j in g) for g in grouped["groups"]]
    by_group = [labels[a] for a in np.argsort(-grouped["mean"], kind="stable")]
    print("ranking from single shuffles:  " + " > ".join(by_single))
    print("ranking with the pair grouped: " + " > ".join(by_group))
    assert pair > s_dec + s_bur, "the pair should carry more than its two single figures"


_try("grouped versus single", _show_grouped_versus_single, needs=("exercise 1", "exercise 2"))

The literature has found the opposite failure as well. Hooker, Mentch and Zhou report that
permute-and-predict measures can *over*-state correlated features, because shuffling one of
a correlated pair forces the model to predict on combinations with little or no data behind
them. Which way the single-feature number goes depends on how the model uses the pair; that
it is not a statement about the pair does not. Section 6 measures the combinations.

## 4. Exercise 3 — `partial_dependence()`

Friedman's partial dependence answers "what does the model predict, on average over the
applicants we have, if this one feature is set to g?" Set the feature to g for every row,
predict, average — once per grid value. The average is over the applicants' OTHER features
as they are, which is exactly what makes it an estimate from the data rather than a guess.

<details><summary>💡 Hint 1 — what to think about</summary>

The average is of predictions, not a prediction at an average. For a model that is not a
straight line — a probability out of a logistic model is not — the prediction for the
average applicant and the average of everyone's predictions are different numbers, and only
one of them is partial dependence. Your caller's data must come back untouched.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate that the data is 2-D, the feature index is in range and the grid is not empty. For
each grid value, take a copy of the data with that one column overwritten by the grid value
in every row, run the model on the whole copy, and store the mean of what comes back.

</details>

In [ ]:
def partial_dependence(model: Callable[[np.ndarray], np.ndarray], X: np.ndarray,
                       feature: int, grid) -> np.ndarray:
    """One-dimensional partial dependence of `model` on column `feature`, over `grid`.

    Requirements, each graded:
      * entry a is the MEAN of `model(X_a)`, where X_a is a copy of X with column `feature`
        set to `grid[a]` in every row — the average of predictions, not the prediction at
        the average row.
      * one entry per grid value, in grid order, as a float array.
      * the caller's X is not changed.
      * `ValueError` if X is not 2-D, `feature` is out of range or `grid` is empty.

    Example:
        >>> X = np.array([[0.0, 1.0], [2.0, 3.0]])
        >>> partial_dependence(lambda A: A[:, 0] * A[:, 1], X, 0, [1.0, 2.0]).tolist()
        [2.0, 4.0]

    Returns:
        numpy float array of length `len(grid)`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_partial_dependence() -> None:
    X = np.array([[0.0, 1.0, 2.0], [1.0, -1.0, 0.5], [2.0, 3.0, -1.0], [-1.0, 0.0, 1.0]])
    before = X.copy()
    model = lambda A: 1.0 / (1.0 + np.exp(-(A[:, 0] * A[:, 1] + A[:, 2])))  # noqa: E731
    grid = [-1.0, 0.0, 2.0]
    got = np.asarray(partial_dependence(model, X, 0, grid), dtype=float)
    assert np.array_equal(X, before), "partial_dependence changed the caller's X — use a copy"
    assert got.shape == (3,), f"one value per grid point: expected shape (3,), got {got.shape}"
    want = []
    for g in grid:
        rows = X.copy()
        rows[:, 0] = g
        want.append(float(np.mean(model(rows))))
    at_mean = []
    for g in grid:
        row = X.mean(axis=0)
        row[0] = g
        at_mean.append(float(model(row[None, :])[0]))
    assert not np.allclose(got, at_mean), (
        "these are predictions at the AVERAGE applicant; partial dependence is the average of "
        "every applicant's prediction with the feature set to the grid value"
    )
    assert np.allclose(got, want, atol=1e-12), (
        f"got {np.round(got, 6).tolist()}, expected {np.round(want, 6).tolist()} — set the "
        "column to the grid value in EVERY row, predict, then average"
    )
    try:
        partial_dependence(model, X, 3, grid)
    except ValueError:
        pass
    else:
        raise AssertionError("a feature index beyond the last column must raise ValueError")
    print("exercise 3 looks right")

In [ ]:
_try("exercise 3", _check_partial_dependence)

In [ ]:
def _show_partial_dependence() -> None:
    print("partial dependence of the champion's probability of default, held-out sample")
    print(f"  {'value':>6}  " + "  ".join(f"{name:>16}" for name in PD_FEATURES))
    curves = {name: partial_dependence(champion_prob, X_HOLD, FEATURES.index(name), GRID)
              for name in PD_FEATURES}
    for a, g in enumerate(GRID):
        print(f"  {g:>+6.1f}  " + "  ".join(f"{curves[name][a]:>16.4f}" for name in PD_FEATURES))


_try("partial dependence", _show_partial_dependence, needs=("exercise 3",))

## 5. Exercise 4 — `partial_dependence_2d()`

The two-dimensional version fixes two features at once, on a grid of pairs. It is how an
interaction is usually looked for — and, for a correlated pair, it is also the plainest
picture of the problem section 6 measures: most of the grid is combinations of declared and
bureau income that nobody in the book has.

<details><summary>💡 Hint 1 — what to think about</summary>

Both features are set in the same copy, at the same time, before the model sees it. Two
one-dimensional curves added together describe a model with no interaction, which is exactly
the thing a two-dimensional plot exists to test. And a 3-by-4 grid has a first axis and a
second axis: be sure which is which.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate as before, and refuse the same feature twice. Loop over the first grid on the
outside and the second on the inside; for each pair, copy the data, overwrite the first
feature with the first value and the second feature with the second, predict, and store the
mean at row "first index", column "second index".

</details>

In [ ]:
def partial_dependence_2d(model: Callable[[np.ndarray], np.ndarray], X: np.ndarray,
                          features, grids) -> np.ndarray:
    """Two-dimensional partial dependence over `grids[0]` x `grids[1]`.

    `features` is a pair of column indices (j, k); `grids` a pair of grids (g1, g2).
    Requirements, each graded:
      * entry [a, b] is the mean of `model` over a copy of X with column j set to g1[a] AND
        column k set to g2[b] in every row — both at once, never the sum of two 1-D curves.
      * shape `(len(g1), len(g2))`: the first grid runs down the rows.
      * the caller's X is not changed.
      * `ValueError` if j == k, either index is out of range, either grid is empty, or X is
        not 2-D.

    Example:
        >>> X = np.array([[0.0, 0.0], [1.0, 1.0]])
        >>> partial_dependence_2d(lambda A: A[:, 0] * A[:, 1], X, (0, 1),
        ...                       ([1.0, 2.0], [3.0, 4.0, 5.0])).tolist()
        [[3.0, 4.0, 5.0], [6.0, 8.0, 10.0]]

    Returns:
        numpy float array of shape `(len(grids[0]), len(grids[1]))`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_partial_dependence_2d() -> None:
    X = np.array([[0.5, 1.0, -1.0], [1.0, -2.0, 0.0], [-1.0, 0.5, 2.0]])
    before = X.copy()
    model = lambda A: A[:, 0] * A[:, 1] + A[:, 2]  # noqa: E731
    g1, g2 = [1.0, 2.0], [0.0, 1.0, 3.0]
    got = np.asarray(partial_dependence_2d(model, X, (0, 1), (g1, g2)), dtype=float)
    assert np.array_equal(X, before), "partial_dependence_2d changed the caller's X"
    assert got.shape == (2, 3), (
        f"shape {got.shape}: the first grid runs down the rows, so it must be (2, 3)"
    )
    mean_z = float(X[:, 2].mean())
    want = np.array([[u * v + mean_z for v in g2] for u in g1])
    assert np.allclose(got, want, atol=1e-12), (
        f"got {np.round(got, 4).tolist()}, expected {np.round(want, 4).tolist()} — set BOTH "
        "features in the same copy before predicting; adding two 1-D curves loses the "
        "interaction a 2-D plot exists to show"
    )
    try:
        partial_dependence_2d(model, X, (1, 1), (g1, g2))
    except ValueError:
        pass
    else:
        raise AssertionError("the same feature twice must raise ValueError")
    print("exercise 4 looks right")

In [ ]:
_try("exercise 4", _check_partial_dependence_2d)

In [ ]:
def _show_pd_2d() -> None:
    surface = partial_dependence_2d(champion_prob, X_HOLD, (0, 1), (GRID_2D, GRID_2D))
    print("probability of default; rows: declared income, columns: bureau income")
    print("        " + "".join(f"{v:>+7.1f}" for v in GRID_2D))
    for a, u in enumerate(GRID_2D):
        print(f"  {u:>+5.1f} " + "".join(f"{surface[a, b]:>7.3f}" for b in range(len(GRID_2D))))


_try("two-dimensional partial dependence", _show_pd_2d, needs=("exercise 4",))

## 6. Exercise 5 — `off_manifold_share()`

Every point partial dependence averages over is a real applicant with ONE feature replaced.
When that feature is correlated with another, the replacement manufactures combinations —
a declared income two standard deviations above the bureau's figure — that the data never
contains. The model will still return a number there. Nothing validated it.

This exercise measures how often that happens. Take the pair (feature, partner), estimate
their mean and covariance from the data, and call a point off the data's manifold when its
Mahalanobis distance puts it outside the ellipse that holds `level` of a bivariate normal.
For two dimensions that ellipse has a closed form: squared distance above `-2 ln(1 - level)`.
The data's own share outside it is the yardstick; partial dependence's share is the finding.

<details><summary>💡 Hint 1 — what to think about</summary>

The distance has to respect the correlation. Plain Euclidean distance puts a point with both
incomes high and a point with declared income high and bureau income low at the same
distance from the centre — yet the first is an ordinary well-off applicant and the second is
nobody in the book. So the inverse of the 2-by-2 covariance matrix sits in the middle of the
distance. The partner values are the rows' own, unless the caller hands you others.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate the level, the two indices and the grid. Compute the two means, the two sample
variances and the covariance with the n minus 1 divisor, and the determinant. Write the
squared distance of a point from the mean using the inverse of that 2-by-2 matrix. Share of
the data outside the threshold first; then, for each grid value, the share of partner values
whose point (grid value, partner value) falls outside it; the overall share is their mean.

</details>

In [ ]:
def off_manifold_share(X: np.ndarray, feature: int, partner: int, grid,
                       level: float = MANIFOLD_LEVEL, partner_values=None) -> dict:
    """How much of partial dependence's evaluation lies outside the data's ellipse.

    Requirements, each graded:
      * the ellipse comes from the data: m = the means of columns `feature` and `partner`, S =
        their 2x2 sample covariance (divisor n - 1). The squared Mahalanobis distance of a
        point u is `(u - m) S^-1 (u - m)^T`.
      * `threshold` is `-2 ln(1 - level)`, the squared radius of the ellipse holding `level`
        of a bivariate normal. A point is off the manifold when its distance EXCEEDS it.
      * `data_share` is the share of the real rows (their own two values) off the manifold.
      * `per_grid[a]` is the share of points `(grid[a], p)` off the manifold, over the partner
        values p — each row's own partner value by default, or `partner_values` if given.
      * `overall` is the mean of `per_grid`: the share of all the evaluation points.
      * `ValueError` if level is not strictly between 0 and 1, feature == partner, either
        index is out of range, the grid is empty, or the covariance is singular.

    Example — 500 independent pairs. At feature = 0 only the few rows whose own partner value
    is extreme fall outside; at feature = 5 every point does:
        >>> r = off_manifold_share(np.random.default_rng(0).normal(size=(500, 2)), 0, 1,
        ...                        [0.0, 5.0])
        >>> round(r["threshold"], 4), r["per_grid"].tolist()
        (9.2103, [0.006, 1.0])

    Returns:
        dict with `threshold`, `data_share`, `overall` (floats) and `per_grid` (float array).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_off_manifold() -> None:
    X = np.array([[0.0, 0.0], [1.0, 1.1], [2.0, 1.9], [-1.0, -0.9], [-2.0, -2.1],
                  [0.5, 0.4], [-0.5, -0.6], [1.5, 1.6]])
    r = off_manifold_share(X, 0, 1, [0.0, 2.0], level=0.9)
    assert abs(r["threshold"] - (-2.0 * math.log(0.1))) < 1e-12, (
        f"threshold {r['threshold']:.6f}: the squared radius of a 2-D normal's level ellipse is "
        "-2 ln(1 - level); a one-dimensional normal quantile is the wrong distribution"
    )
    pg = np.asarray(r["per_grid"], dtype=float)
    assert pg.shape == (2,), f"per_grid needs one share per grid value, got shape {pg.shape}"
    assert pg[1] > 0.7, (
        f"at feature = 2.0 only {pg[1]:.3f} of the points were called off-manifold; with the "
        "pair this tightly correlated, (2.0, partner) is far off the diagonal for nearly every "
        "row — did you use the identity instead of the inverse covariance?"
    )
    assert abs(r["data_share"]) < 1e-12, (
        "every real row sits on the diagonal here, so none of them is off the manifold"
    )
    assert abs(r["overall"] - float(pg.mean())) < 1e-12, "overall is the mean of per_grid"
    fixed = off_manifold_share(X, 0, 1, [2.0], level=0.9, partner_values=[2.0, -2.0])
    assert np.allclose(fixed["per_grid"], [0.5]), (
        "with partner_values given, per_grid is taken over THOSE values: (2, 2) lies on the "
        "diagonal and (2, -2) far off it, so the share is one half"
    )
    for bad in ({"level": 1.0}, {"level": 0.0}):
        try:
            off_manifold_share(X, 0, 1, [0.0], **bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"level={bad['level']} must raise ValueError")
    print("exercise 5 looks right")

In [ ]:
_try("exercise 5", _check_off_manifold)

In [ ]:
def _show_off_manifold() -> None:
    inc = off_manifold_share(X_HOLD, 0, 1, GRID)
    ctl = off_manifold_share(X_HOLD, 2, 0, GRID)
    print(f"ellipse holding {MANIFOLD_LEVEL} of a bivariate normal; "
          f"squared-distance threshold {inc['threshold']:.4f}")
    print(f"real applicants outside it: income pair {inc['data_share']:.3f}, "
          f"utilisation with declared income {ctl['data_share']:.3f}\n")
    print("share of partial dependence's evaluation points outside the ellipse")
    print(f"  {'value':>6}  {'declared (vs bureau)':>21}  {'utilisation (vs declared)':>26}")
    for a, g in enumerate(GRID):
        print(f"  {g:>+6.1f}  {inc['per_grid'][a]:>21.3f}  {ctl['per_grid'][a]:>26.3f}")
    print(f"  {'all':>6}  {inc['overall']:>21.3f}  {ctl['overall']:>26.3f}")
    shuffled = off_manifold_share(X_HOLD, 0, 1, X_HOLD[:, 0])
    print(f"\nshuffling declared income (exercise 1) pairs every declared value with every "
          f"bureau value;\na share of {shuffled['overall']:.3f} of those pairings lies outside "
          "the ellipse")
    cells = off_manifold_share(X_HOLD, 0, 1, GRID_2D, partner_values=GRID_2D)
    print(f"of the {len(GRID_2D) ** 2} cells in the two-dimensional grid of section 5, a share "
          f"of {cells['overall']:.3f} lies outside it")


_try("off the manifold", _show_off_manifold, needs=("exercise 5",))

A share is not yet a consequence. Here is one. Fit a second model on the same development
rows that uses bureau income and never looks at declared income. On the applicants that
exist, the two models rank almost identically and predict much alike, because on those
applicants the two incomes agree. Partial dependence on declared income asks what happens
when they disagree — and the two models give different answers to a question the data
cannot settle. Compare the gap on real applicants with the gap between the curves.

In [ ]:
def _show_two_models() -> None:
    alt = fit_logistic(BOOK["X_dev"][:, 1:], BOOK["y_dev"])

    def alt_prob(X: np.ndarray) -> np.ndarray:
        return champion_probability(alt, np.asarray(X)[:, 1:])

    auc_a = auc_by_ranks(Y_HOLD, champion_score(X_HOLD))
    auc_b = auc_by_ranks(Y_HOLD, champion_logit(alt, X_HOLD[:, 1:]))
    gaps = np.abs(champion_prob(X_HOLD) - alt_prob(X_HOLD))
    print(f"held-out AUC: champion {auc_a:.4f}, bureau-only model {auc_b:.4f}")
    print(f"gap between their predictions on the real held-out applicants: mean "
          f"{float(gaps.mean()):.4f}, median {float(np.median(gaps)):.4f}\n")
    pd_a = partial_dependence(champion_prob, X_HOLD, 0, GRID)
    pd_b = partial_dependence(alt_prob, X_HOLD, 0, GRID)
    om = off_manifold_share(X_HOLD, 0, 1, GRID)
    print("partial dependence on DECLARED income")
    print(f"  {'value':>6}  {'champion':>9}  {'bureau-only':>11}  {'gap':>7}  {'off-manifold':>12}")
    for a, g in enumerate(GRID):
        print(f"  {g:>+6.1f}  {pd_a[a]:>9.4f}  {pd_b[a]:>11.4f}  {pd_a[a] - pd_b[a]:>+7.4f}"
              f"  {om['per_grid'][a]:>12.3f}")
    edge = float(max(abs(pd_a[0] - pd_b[0]), abs(pd_a[-1] - pd_b[-1])))
    print(f"\nat the edges of the grid their curves are {edge:.4f} apart. One model says "
          "declared income\nmatters a great deal, the other that it does not matter at all. "
          "Both fit the data;\nthe disagreement lives where the data does not.")


_try("two models", _show_two_models, needs=("exercise 3", "exercise 5"))

## 7. Exercise 6 — `coalition_value()`

A local attribution explains ONE applicant's prediction: how much of the gap between this
applicant and a baseline each feature accounts for. Shapley's value — built for sharing a
coalition's worth among its players, and pinned down by three axioms — needs a *value
function*: what the model predicts when only the features in a coalition S are known.

There is no single right value function, so the evidence has to state which one it used.
This lesson's is **interventional over a background sample**: take each background row, overwrite
the features in S with the applicant's values, predict, and average over the rows. With S
empty that is the model's average prediction on the background — the baseline value. With
every feature in S it is the applicant's own prediction.

<details><summary>💡 Hint 1 — what to think about</summary>

"Features not in S are unknown" is represented by the background applicants' own values, row
by row — not by a single average applicant, and not by zeros. The average of predictions and
the prediction at the average differ for a probability model, and only one of them is this
value function. The value function will be called many times; the background must be the
same on every call.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate the shapes, keep your own copy of the background, and return a small function of a
coalition. Inside it: copy the background, overwrite the coalition's columns in every row with
the applicant's values for those columns, run the model and return the mean as a float.

</details>

In [ ]:
def coalition_value(model: Callable[[np.ndarray], np.ndarray], x: np.ndarray,
                    background: np.ndarray) -> Callable[[tuple], float]:
    """The interventional value function for explaining the prediction at `x`.

    Returns `value(coalition)`, where `coalition` is a tuple of feature indices. Requirements,
    each graded:
      * `value(S)` is the mean over background rows of `model(row with the columns in S
        replaced by x's values)`, as a float. Features outside S keep each background row's
        own values — never an average row, never zero.
      * `value(())` is therefore the mean prediction on the background (the baseline value)
        and `value(all features)` is `model(x)`.
      * the background is the same on every call: no call may change what a later one sees,
        and the caller's arrays are not changed.
      * `ValueError` if background is not 2-D with at least one row, or x does not match its
        columns.

    Example — the two background rows predict 1 and 0, so the baseline value is their mean:
        >>> v = coalition_value(lambda A: A[:, 0] * A[:, 1], np.array([2.0, 3.0]),
        ...                     np.array([[1.0, 1.0], [0.0, 5.0]]))
        >>> v(()), v((0,)), v((0, 1))
        (0.5, 6.0, 6.0)

    Returns:
        a function from a tuple of feature indices to a float.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_coalition_value() -> None:
    model = lambda A: 1.0 / (1.0 + np.exp(-(A[:, 0] * A[:, 1] - A[:, 2])))  # noqa: E731
    bg = np.array([[0.0, 1.0, 0.5], [2.0, -1.0, 0.0], [1.0, 3.0, -1.0], [-1.0, 0.0, 2.0]])
    x = np.array([1.5, 2.0, -0.5])
    bg_before = bg.copy()
    v = coalition_value(model, x, bg)
    empty = v(())
    assert abs(empty - float(np.mean(model(bg)))) < 1e-12, (
        f"value(()) = {empty:.6f}: with no features known it is the MEAN prediction over the "
        "background rows — not the prediction at the mean row, and not a row of zeros"
    )
    assert abs(v((0, 1, 2)) - float(model(x[None, :])[0])) < 1e-12, (
        "value(all features) must be the model's prediction for x itself"
    )
    rows = bg.copy()
    rows[:, [0, 2]] = x[[0, 2]]
    assert abs(v((0, 2)) - float(np.mean(model(rows)))) < 1e-12, (
        "value((0, 2)) replaces columns 0 and 2 with x's values in every background row and "
        "keeps each row's own column 1"
    )
    assert abs(v(()) - empty) < 1e-12 and np.array_equal(bg, bg_before), (
        "a call changed the background — later calls saw different rows; copy before writing"
    )
    print("exercise 6 looks right")

In [ ]:
_try("exercise 6", _check_coalition_value)

## 8. Exercise 7 — `shapley_values()`

The Shapley attribution of feature i averages its marginal contribution `v(S ∪ {i}) - v(S)`
over every coalition S that leaves it out, weighted by `|S|! (p - |S| - 1)! / p!` — the share
of all p! orderings of the features in which exactly S arrives before i. Exact enumeration
visits every one of the 2^p coalitions. The end of this section times what that costs.

The axiom that makes the result an *explanation* is efficiency: the attributions add up to
`v(all) - v(())`, the applicant's prediction minus the baseline. Nothing is left over and
nothing is counted twice.

<details><summary>💡 Hint 1 — what to think about</summary>

The value function is the expensive part: every call is a batch of model predictions. There
are 2^p coalitions and each marginal contribution needs two of them, but every coalition is
needed by several features — so evaluate each one once and look it up after. The weight
depends only on the SIZE of the coalition you are adding to. Coalitions go to the value
function as tuples of indices in ascending order.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Enumerate every subset of the p features once — counting through the integers below 2^p and
reading each one's bits as membership is the tidiest way — call the value function on each as
an ascending tuple, and store the answer against the subset. Then, for each feature, sum over
the subsets that exclude it: the weight for that subset's size times the difference between
the subset with the feature added and the subset itself.

</details>

In [ ]:
def shapley_values(value: Callable[[tuple], float], n_features: int) -> np.ndarray:
    """Exact Shapley values of `n_features` players under `value`, by full enumeration.

    Requirements, each graded:
      * `phi[i] = sum over S not containing i of |S|! (p-|S|-1)! / p! * (v(S+{i}) - v(S))`,
        exactly, for every i.
      * `value` is called exactly once for each of the 2^p coalitions, each passed as a tuple
        of indices in ascending order (`()` for the empty coalition).
      * efficiency follows: `phi.sum() == value(all) - value(())` to rounding.
      * `ValueError` if n_features < 1.

    Not graded, but the notebook depends on it: the work must grow like 2^p, not p!. Averaging
    over every ordering of the features gives the same numbers, but the cost and sampling cells
    below run this on more features than p! orderings can finish in any sensible time.

    Example — a two-player game worth 0 alone, 10 to player 0 alone, 4 to player 1 alone and
    20 together; player 0 gets (10 + (20 - 4)) / 2:
        >>> table = {(): 0.0, (0,): 10.0, (1,): 4.0, (0, 1): 20.0}
        >>> shapley_values(lambda s: table[s], 2).tolist()
        [13.0, 7.0]

    Returns:
        numpy float array of length `n_features`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_shapley() -> None:
    table = {(): 0.0, (0,): 10.0, (1,): 4.0, (0, 1): 20.0}
    got = np.asarray(shapley_values(lambda s: table[s], 2), dtype=float)
    assert np.allclose(got, [13.0, 7.0]), (
        f"got {got.tolist()} for the docstring's two-player game, expected [13.0, 7.0]"
    )
    # Three players: a glove game. Player 0 holds a left glove, players 1 and 2 a right one
    # each; a coalition is worth 1 if it can make a pair. Shapley gives 2/3, 1/6, 1/6.
    glove = lambda s: 1.0 if 0 in s and (1 in s or 2 in s) else 0.0  # noqa: E731
    g = np.asarray(shapley_values(glove, 3), dtype=float)
    assert np.allclose(g, [2 / 3, 1 / 6, 1 / 6]), (
        f"glove game gave {np.round(g, 4).tolist()}, expected [0.6667, 0.1667, 0.1667] — check "
        "the weights: |S|!(p-|S|-1)!/p!, which sum to 1 over the coalitions; 1/2^(p-1) is the "
        "Banzhaf index and does not add up to the prediction"
    )
    rng = np.random.default_rng(5)
    worth = {s: float(rng.normal()) for s in
             [tuple(i for i in range(4) if m >> i & 1) for m in range(16)]}
    calls = []

    def spy(s: tuple) -> float:
        calls.append(s)
        return worth[tuple(s)]

    phi = np.asarray(shapley_values(spy, 4), dtype=float)
    assert abs(phi.sum() - (worth[(0, 1, 2, 3)] - worth[()])) < 1e-9, (
        "efficiency fails: the attributions must add up to v(all) - v(())"
    )
    assert len(calls) == 16 and len(set(map(tuple, calls))) == 16, (
        f"the value function was called {len(calls)} times for 16 coalitions — evaluate each "
        "coalition once and look it up; each call is a batch of model predictions"
    )
    assert all(list(c) == sorted(c) for c in calls), "pass each coalition as an ASCENDING tuple"
    print("exercise 7 looks right")

In [ ]:
_try("exercise 7", _check_shapley)

In [ ]:
def shapley_applicants(prob_hold: np.ndarray) -> tuple:
    """The three held-out applicants the pack explains: low, middle and high predicted risk.

    Given to you. Picked by rank, not by hand, so the choice regenerates with the pack.
    """
    order = np.argsort(prob_hold, kind="stable")
    n = order.size
    return tuple(int(order[k]) for k in (n // 20, n // 2, n - 1 - n // 20))


def _show_applicants() -> None:
    background = BOOK["X_dev"][:N_BACKGROUND]
    picks = shapley_applicants(champion_prob(X_HOLD))
    print(f"value function: interventional, mean over the first {N_BACKGROUND} development "
          "rows")
    print(f"  {'':<16}" + "".join(f"{'applicant ' + str(i):>16}" for i in picks))
    columns = []
    for i in picks:
        value = coalition_value(champion_prob, X_HOLD[i], background)
        phi = shapley_values(value, len(FEATURES))
        base, pred = value(()), value(tuple(range(len(FEATURES))))
        columns.append((base, phi, pred))
        assert abs(float(phi.sum()) - (pred - base)) < 1e-12, "efficiency must hold"
    print(f"  {'baseline v(())':<16}" + "".join(f"{c[0]:>16.4f}" for c in columns))
    for j, name in enumerate(FEATURES):
        print(f"  {name:<16}" + "".join(f"{c[1][j]:>+16.4f}" for c in columns))
    print(f"  {'prediction':<16}" + "".join(f"{c[2]:>16.4f}" for c in columns))
    print("\nEach column adds up: baseline plus the six attributions is the prediction,")
    print("to within 1e-12, for every applicant — the efficiency axiom, checked, not assumed.")


_try("three applicants", _show_applicants, needs=("exercise 6", "exercise 7"))

### What exact enumeration costs

The pack's handful of features costs nothing to enumerate. The cell below times your
`shapley_values` on a synthetic model with a growing number of features, prints each time
with its unit, and extrapolates from the growth it measured. Your times will differ from
anyone else's. The coalitions double with every feature on every machine, and the time
cannot grow more slowly than they do.

In [ ]:
def _timing_game(p: int):
    """A synthetic probability model on p features with interactions, and an applicant."""
    rng = np.random.default_rng(SEED + p)
    w = rng.normal(0.0, 0.6, p)
    background = rng.normal(0.0, 1.0, (16, p))
    x = rng.normal(0.0, 1.0, p)

    def model(A: np.ndarray) -> np.ndarray:
        z = A[:, 0] * A[:, -1] * 0.3
        for j in range(p):
            z = z + w[j] * A[:, j]
        return 1.0 / (1.0 + np.exp(-z))

    return model, x, background


def _show_cost() -> None:
    times = {}
    print(f"  {'features':>8}  {'coalitions':>10}  {'time':>12}")
    for p in range(2, 15):
        model, x, background = _timing_game(p)
        value = coalition_value(model, x, background)
        t0 = time.perf_counter()
        shapley_values(value, p)
        times[p] = time.perf_counter() - t0
        print(f"  {p:>8}  {2 ** p:>10,}  {times[p] * 1000:>9.3f} ms")
    growth = (times[14] / times[11]) ** (1.0 / 3.0)
    at_30 = times[14] * growth ** 16
    at_60 = times[14] * growth ** 46
    print(f"\neach extra feature multiplied the time by ≈{growth:.2f} over the last three steps")
    print(f"at that rate, one applicant with 30 features: ≈ {at_30 / 3600:,.0f} hours")
    print(f"and with 60 features: ≈ {at_60 / (3600 * 24 * 365.25):,.0f} years")


_try("cost of enumeration", _show_cost, needs=("exercise 6", "exercise 7"))

That wall is why Štrumbelj and Kononenko, explaining individual predictions with feature
contributions drawn from coalitional game theory, overcame the method's exponential cost
with a sampling-based approximation. The simplest version of the idea draws random
orderings of the features and averages each feature's marginal contribution along them.
The cell below compares it with your exact answer on a larger synthetic model. Read the
error columns against the last one: whether the estimate still sums to the prediction minus
the baseline.

In [ ]:
def sampled_shapley(value: Callable[[tuple], float], n_features: int, n_orderings: int,
                    rng: np.random.Generator) -> tuple:
    """Shapley values estimated from `n_orderings` random orderings. Given to you.

    Returns (estimate, number of value calls). It is random, so it takes a generator — and
    an estimate that is evidence records the seed it was drawn with.
    """
    phi = np.zeros(n_features)
    calls = 0
    for _ in range(n_orderings):
        order = rng.permutation(n_features)
        members: list = []
        before = float(value(()))
        calls += 1
        for i in order:
            members.append(int(i))
            after = float(value(tuple(sorted(members))))
            calls += 1
            phi[int(i)] += after - before
            before = after
    return phi / n_orderings, calls


def _show_sampling() -> None:
    p = 12
    model, x, background = _timing_game(p)
    value = coalition_value(model, x, background)
    exact = shapley_values(value, p)
    target = value(tuple(range(p))) - value(())
    print(f"exact enumeration: {2 ** p:,} value calls")
    print(f"  {'orderings':>9}  {'value calls':>11}  {'worst error':>11}  "
          f"{'coordinates off by > 1e-6':>25}  {'sums to f(x) - v(())':>20}")
    for m in (8, 64, 512):
        est, calls = sampled_shapley(value, p, m, np.random.default_rng(SEED))
        errors = np.abs(est - exact)
        off = int((errors > 1e-6).sum())
        adds_up = abs(float(est.sum()) - target) < 1e-12
        print(f"  {m:>9}  {calls:>11,}  {float(errors.max()):>11.5f}  {f'{off} of {p}':>25}"
              f"  {str(adds_up):>20}")
    print("\nThe sampled estimate adds up exactly while its coordinates are off. Efficiency is")
    print("necessary evidence that an attribution is right, never sufficient evidence.")


_try("sampling", _show_sampling, needs=("exercise 6", "exercise 7"))

## 9. Exercise 8 — `canonical_bytes()`

Everything so far goes into one explanation pack: a dict of results. To say two packs are
the same you need one fixed way of writing a pack down, so that equal content always makes
equal bytes and unequal content never does. That is a canonical serialisation, and the
certificate in section 12 is a hash of it.

Two failures matter. Writing keys in the order they happened to be inserted makes equal packs
look different — a check that cries wolf gets switched off. Rounding makes different packs
look equal — a check that is blind to the last digit certifies whatever it cannot see.

<details><summary>💡 Hint 1 — what to think about</summary>

JSON can do nearly all of it, if you ask it to: sorted keys, one value per line, and no
pretending that NaN is a number. What JSON cannot do is read numpy — so turn arrays and numpy
scalars into plain Python lists and numbers first, without passing through anything that
rounds. Python's own float formatting already writes the shortest text that reads back to the
same bits.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Write a small recursive converter: dictionaries to dictionaries with string keys, lists,
tuples and arrays to lists, numpy scalars to the matching Python scalar, and anything else
refused. Then dump the result as JSON with keys sorted, an indent so every scalar sits on its
own line, and non-finite numbers refused; add a final newline and encode it.

</details>

In [ ]:
def canonical_bytes(pack: dict) -> bytes:
    """Serialise `pack` so equal content gives equal bytes and unequal content never does.

    Requirements, each graded:
      * returns `bytes` that `json.loads` reads back to the same content: dict keys are
        strings, numpy arrays and tuples become lists, numpy scalars become Python numbers.
      * keys are sorted at every level, so the order a pack was assembled in cannot change
        its bytes.
      * floats are written exactly — two floats that differ in the last bit give different
        bytes. Never round, never cast to float32.
      * one scalar per line (JSON with an indent does this), so a divergence can be located
        by line number.
      * `ValueError` for NaN or infinity anywhere in the pack: a non-finite number in evidence
        is a finding to fix, and NaN is not even equal to itself.

    Example:
        >>> canonical_bytes({"b": np.float64(0.5), "a": (1, 2)}) == canonical_bytes(
        ...     {"a": [1, 2], "b": 0.5})
        True

    Returns:
        bytes (UTF-8 JSON text ending in a newline).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_canonical() -> None:
    a = canonical_bytes({"z": 1.0, "a": {"y": 2, "b": [1, 2]}})
    b = canonical_bytes({"a": {"b": [1, 2], "y": 2}, "z": 1.0})
    assert isinstance(a, bytes), f"return bytes, not {type(a).__name__} — encode the text"
    assert a == b, (
        "the same content assembled in a different order gave different bytes — sort the keys "
        "at every level"
    )
    x = 0.1 + 0.2
    assert canonical_bytes({"v": x}) != canonical_bytes({"v": 0.3}), (
        "0.1 + 0.2 and 0.3 differ in the last bit and must give different bytes — do not round"
    )
    assert json.loads(canonical_bytes({"v": x}))["v"] == x, "floats must read back exactly"
    c = canonical_bytes({"arr": np.array([[1.5, 2.5]]), "n": np.int64(3), "f": np.float64(0.25),
                         "t": (1, 2)})
    assert json.loads(c) == {"arr": [[1.5, 2.5]], "n": 3, "f": 0.25, "t": [1, 2]}, (
        "numpy arrays, numpy scalars and tuples must become plain lists and numbers"
    )
    for bad in (float("nan"), np.array([1.0, np.inf])):
        try:
            canonical_bytes({"v": bad})
        except ValueError:
            pass
        else:
            raise AssertionError("NaN or infinity in a pack must raise ValueError")
    many = canonical_bytes({f"k{i}": float(i) for i in range(6)})
    assert len(many.decode().strip().splitlines()) >= 6, (
        "every scalar needs its own line, or the certificate cannot say where two packs part "
        "company — dump with an indent"
    )
    print("exercise 8 looks right")

In [ ]:
_try("exercise 8", _check_canonical)

## 10. The pack, and a process that is not this one

Below is the first line's explanation script, as delivered: `assemble_pack` builds every
result in this notebook from the seed alone — the data, the model, the importances, the
curves, the manifold shares and the three applicants' attributions — and `first_line_pack` is
the entry point they run. Read both.

To regenerate the pack somewhere that is not this notebook, `regenerate_in_fresh_process`
starts a new Python interpreter, ships it the compiled code of the builder and of every
function it calls (compiled code needs no source file, so this works the same in a notebook
as in a script), plus the simple constants they read, and returns what that interpreter
built. It refuses a builder that reads anything
it cannot rebuild from code and seed — a DataFrame left in a notebook global, say — because
a pack that depends on one is not regenerable. It sets the new process's **PYTHONHASHSEED**,
the seed Python uses to salt the hashes of strings; the reason that matters is exercise 9.

In [ ]:
def assemble_pack(seed: int, asked) -> dict:
    """Build the whole explanation pack from `seed`. Given to you: the first line's code.

    `asked` is the features the committee asked about, in the order they are scored.
    """
    book = make_book(seed)
    coef = fit_logistic(book["X_dev"], book["y_dev"])
    score = lambda X: champion_logit(coef, X)            # noqa: E731
    prob = lambda X: champion_probability(coef, X)       # noqa: E731
    Xh, yh = book["X_hold"], book["y_hold"]
    rng = np.random.default_rng(seed)
    pack = {"pack.seed": seed, "pack.data_note": DATA_NOTE, "pack.model": MODEL_NOTE,
            "pack.features": list(FEATURES), "model.intercept": float(coef[0]),
            "model.holdout_auc": auc_by_ranks(yh, score(Xh))}
    for j, name in enumerate(FEATURES):
        pack[f"model.coef.{name}"] = float(coef[j + 1])
    columns = [FEATURES.index(name) for name in asked]
    single = permutation_importance(score, auc_by_ranks, Xh, yh, N_REPEATS, rng, columns=columns)
    pack["importance.n_repeats"] = N_REPEATS
    pack["importance.baseline_auc"] = single["baseline"]
    for a, j in enumerate(single["columns"]):
        pack[f"importance.{FEATURES[j]}.mean"] = float(single["mean"][a])
        pack[f"importance.{FEATURES[j]}.std"] = float(single["std"][a])
    grouped = grouped_permutation_importance(score, auc_by_ranks, Xh, yh, IMPORTANCE_GROUPS,
                                             N_REPEATS, rng)
    for a, g in enumerate(IMPORTANCE_GROUPS):
        label = "+".join(FEATURES[j] for j in g)
        pack[f"grouped.{label}.mean"] = float(grouped["mean"][a])
        pack[f"grouped.{label}.std"] = float(grouped["std"][a])
    pack["pd.grid"] = list(GRID)
    for name in PD_FEATURES:
        pack[f"pd.{name}"] = partial_dependence(prob, Xh, FEATURES.index(name), GRID)
    pack["pd2d.grid"] = list(GRID_2D)
    pack["pd2d.declared_income|bureau_income"] = partial_dependence_2d(
        prob, Xh, (0, 1), (GRID_2D, GRID_2D))
    pack["manifold.level"] = MANIFOLD_LEVEL
    for j, k in MANIFOLD_PAIRS:
        om = off_manifold_share(Xh, j, k, GRID)
        label = f"{FEATURES[j]}|{FEATURES[k]}"
        pack[f"manifold.{label}.data_share"] = om["data_share"]
        pack[f"manifold.{label}.per_grid"] = om["per_grid"]
        pack[f"manifold.{label}.overall"] = om["overall"]
    pack["manifold.pd2d_cells.overall"] = off_manifold_share(
        Xh, 0, 1, GRID_2D, partner_values=GRID_2D)["overall"]
    background = book["X_dev"][:N_BACKGROUND]
    pack["shapley.value_function"] = (f"interventional: mean prediction over the first "
                                      f"{N_BACKGROUND} development rows")
    for i in shapley_applicants(prob(Xh)):
        value = coalition_value(prob, Xh[i], background)
        phi = shapley_values(value, len(FEATURES))
        pack[f"shapley.applicant_{i}.baseline"] = value(())
        pack[f"shapley.applicant_{i}.prediction"] = value(tuple(range(len(FEATURES))))
        for j, name in enumerate(FEATURES):
            pack[f"shapley.applicant_{i}.{name}"] = float(phi[j])
    return pack


def first_line_pack(seed: int) -> dict:
    """The explanation pack exactly as the first line's script builds it. Given to you."""
    asked = set(COMMITTEE_ASKED)          # the request names utilisation twice: de-duplicate
    return assemble_pack(seed, asked)


_SIMPLE_TYPES = (type(None), bool, int, float, complex, str, bytes)


def _is_simple_constant(v: Any) -> bool:
    if isinstance(v, _SIMPLE_TYPES) or isinstance(v, np.generic):
        return True
    if isinstance(v, (tuple, frozenset)):
        return all(_is_simple_constant(item) for item in v)
    return False


def _global_names(code: types.CodeType) -> set:
    """Every global name a code object, or any function nested inside it, reads."""
    names = set()
    for ins in dis.get_instructions(code):
        if ins.opname in ("LOAD_GLOBAL", "LOAD_NAME", "LOAD_FROM_DICT_OR_GLOBALS"):
            names.add(ins.argval)
    for const in code.co_consts:
        if isinstance(const, types.CodeType):
            names |= _global_names(const)
    return names


def _importable(module_name: str) -> bool:
    root = (module_name or "").split(".")[0]
    return root == "numpy" or root in sys.stdlib_module_names


def _closure_payload(entry: Callable) -> dict:
    """What a fresh interpreter needs to call `entry`: code, constants, and module names."""
    functions, constants, modules, refs = [], {}, {}, {}
    bound: dict = {}
    todo = [("__entry__", entry)]
    while todo:
        alias, fn = todo.pop()
        if alias in bound:
            if bound[alias] is not fn:
                raise ValueError(f"two functions disagree about what `{alias}` means")
            continue
        bound[alias] = fn
        if not isinstance(fn, types.FunctionType):
            raise ValueError(f"`{alias}` is not a plain function and cannot be shipped")
        if fn.__closure__:
            raise ValueError(f"`{fn.__name__}` reads variables of an enclosing function; "
                             "define it at the top level so a fresh process can rebuild it")
        defaults = tuple(fn.__defaults__ or ()) + tuple((fn.__kwdefaults__ or {}).values())
        for d in defaults:
            if not _is_simple_constant(d):
                raise ValueError(f"`{fn.__name__}` has a default argument of type "
                                 f"{type(d).__name__}; use None and build it inside")
        functions.append((alias, marshal.dumps(fn.__code__), fn.__defaults__, fn.__kwdefaults__))
        for name in sorted(_global_names(fn.__code__)):
            if name not in fn.__globals__:
                continue                       # a builtin, or a name that is simply unbound
            obj = fn.__globals__[name]
            if isinstance(obj, types.ModuleType):
                modules[name] = obj.__name__
            elif isinstance(obj, types.FunctionType) and not _importable(obj.__module__):
                todo.append((name, obj))
            elif _is_simple_constant(obj):
                constants[name] = obj
            elif callable(obj) and _importable(getattr(obj, "__module__", "")):
                refs[name] = (obj.__module__, obj.__qualname__)
            else:
                raise ValueError(
                    f"`{fn.__name__}` reads the global `{name}` (a {type(obj).__name__}), which "
                    "a fresh process cannot rebuild from code and seed — regenerate it inside "
                    "the builder, or pass it in as a simple constant")
    return {"functions": functions, "constants": constants, "modules": modules, "refs": refs}


_FRESH_PROCESS = r'''
import builtins, importlib, marshal, pickle, sys, types
payload = pickle.loads(sys.stdin.buffer.read())
g = {"__builtins__": builtins, "__name__": "__fresh_process__"}
for alias, name in payload["modules"].items():
    g[alias] = importlib.import_module(name)
for alias, (module, qualname) in payload["refs"].items():
    obj = importlib.import_module(module)
    for part in qualname.split("."):
        obj = getattr(obj, part)
    g[alias] = obj
g.update(payload["constants"])
for alias, code, defaults, kwdefaults in payload["functions"]:
    fn = types.FunctionType(marshal.loads(code), g, alias, defaults)
    fn.__kwdefaults__ = kwdefaults
    g[alias] = fn
result_stream = sys.stdout.buffer
sys.stdout = sys.stderr          # anything the builder prints must not corrupt the result
result = g["__entry__"](*payload["args"])
result_stream.write(pickle.dumps(result))
result_stream.flush()
'''


def regenerate_in_fresh_process(build: Callable[[int], dict], seed: int, hash_seed: int,
                                timeout: float = 60.0) -> dict:
    """Run `build(seed)` in a brand-new Python interpreter and return what it built.

    Given to you, not graded. The new interpreter is this one (`sys.executable`), started
    with PYTHONHASHSEED set to `hash_seed`, so it hashes strings differently from this
    process and from a process with another seed. Nothing leaves this machine. Raises
    RuntimeError, with the tail of the child's error, if the regeneration fails: a pack that
    could not be rebuilt has not been reproduced.
    """
    if isinstance(hash_seed, bool) or not isinstance(hash_seed, int) \
            or not 0 <= hash_seed <= 4294967295:
        raise ValueError(f"hash_seed must be an integer in [0, 4294967295], got {hash_seed!r}")
    payload = _closure_payload(build)
    payload["args"] = (seed,)
    env = dict(os.environ)
    env["PYTHONHASHSEED"] = str(hash_seed)
    proc = subprocess.run([sys.executable, "-c", _FRESH_PROCESS], input=pickle.dumps(payload),
                          capture_output=True, env=env, timeout=timeout)
    if proc.returncode != 0:
        tail = proc.stderr.decode("utf-8", "replace").strip().splitlines()[-3:]
        raise RuntimeError(f"the fresh process (PYTHONHASHSEED={hash_seed}) failed: "
                           + " | ".join(tail))
    return pickle.loads(proc.stdout)

## 11. Exercise 9 — `reproducibility_check()`

The check regenerates the pack and compares bytes. The subtlety is WHERE it regenerates.
Built twice in this process — this notebook's kernel — a pack shares everything the process
has: the same imported modules, the same state and the same hash seed. Per the Python
documentation, string hashes are salted with a value that stays constant within a process
but is not predictable between invocations, and changing hash values changes the iteration
order of sets. A check that never leaves the process cannot see any of that.

So: build the pack here once — that is the pack in hand, the thing being certified — then
regenerate it in one fresh process per hash seed, and require every one to match it byte for
byte. The certificate records what was compared, what matched, and where the first mismatch
was.

<details><summary>💡 Hint 1 — what to think about</summary>

Each fresh run is compared against the pack in hand, not just against the other fresh runs:
fresh processes that agree with each other and not with what you are about to file have
certified the wrong artefact. Compare the serialised bytes and nothing else — any tolerance
is a place for a difference to hide. And a regeneration that crashed has reproduced nothing:
it must never count as a match.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Refuse fewer than two distinct hash seeds. Serialise the in-process build with the serialiser
you were given (the canonical one if none was). For each hash seed, regenerate in a fresh
process, serialise, record its SHA-256 and whether the bytes are equal. At the first mismatch,
split both texts into lines and record the first line number, counting from one, where they
differ, with both lines. Reproduced means every run matched.

</details>

In [ ]:
def reproducibility_check(build: Callable[[int], dict], seed: int,
                          hash_seeds=HASH_SEEDS, serialise=None) -> dict:
    """Regenerate `build(seed)` in fresh processes and certify byte equality — or refuse to.

    `serialise` defaults to `canonical_bytes` (looked up when called, so an edited exercise 8
    is the one used). Requirements, each graded:
      * the pack in hand is `serialise(build(seed))`, built once in THIS process.
      * for every hash seed, in order: `regenerate_in_fresh_process(build, seed, hash_seed)`,
        serialised the same way, compared with the pack in hand by exact byte equality.
      * `ValueError` if `hash_seeds` holds fewer than two distinct values: one fresh process
        cannot tell a stable pack from a lucky one.
      * a regeneration that raises is never a match — let the error propagate.
      * the certificate is a dict:
          `reproduced`        True only if every fresh run matched the pack in hand
          `seed`              the seed
          `sha256`            hex SHA-256 of the pack in hand's bytes
          `n_bytes`           their length
          `hash_seeds`        the hash seeds, as a list, in order
          `runs`              one dict per hash seed, in order: `hash_seed`, `sha256`, `match`
          `first_difference`  None, or for the FIRST mismatching run: `hash_seed`, `line`
                              (1-based line number of the first differing line), `expected`
                              (that line in the pack in hand) and `got` (that line in the run);
                              a missing line reads as ""
          `environment`       {"python": platform.python_version(), "numpy": np.__version__}

    Example:
        >>> cert = reproducibility_check(lambda s: {"seed": s}, 7)   # doctest: +SKIP
        >>> cert["reproduced"], [r["match"] for r in cert["runs"]]  # doctest: +SKIP
        (True, [True, True, True])

    Returns:
        the certificate dict described above.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _stable_build(seed: int) -> dict:
    """A tiny builder with nothing to hide. Used by the check below."""
    rng = np.random.default_rng(seed)
    return {"draw": float(rng.random()), "seed": seed}


def _set_order_build(seed: int) -> dict:
    """A tiny builder whose numbers depend on the order a set of strings is visited in."""
    rng = np.random.default_rng(seed)
    out = {}
    for name in set(("arrears", "enquiries", "income", "tenure", "utilisation")):
        out[name] = float(rng.random())
    return out


def _check_reproducibility() -> None:
    try:
        reproducibility_check(_stable_build, 3, hash_seeds=(4, 4))
    except ValueError:
        pass
    else:
        raise AssertionError("two identical hash seeds are one experiment: raise ValueError")
    cert = reproducibility_check(_stable_build, 3, serialise=canonical_bytes)
    assert cert["reproduced"] is True, (
        "a builder with no hidden dependence came back NOT reproduced — compare the serialised "
        "bytes of each fresh run with the pack in hand"
    )
    want = hashlib.sha256(canonical_bytes(_stable_build(3))).hexdigest()
    assert cert["sha256"] == want, (
        "sha256 must be the SHA-256 of serialise(build(seed)) — the bytes of the pack in hand"
    )
    assert [r["hash_seed"] for r in cert["runs"]] == list(HASH_SEEDS), (
        "one run per hash seed, in order, each recording the hash seed it used"
    )
    assert cert["first_difference"] is None, "a reproduced pack has no first difference"
    bad = reproducibility_check(_set_order_build, 3, serialise=canonical_bytes)
    assert bad["reproduced"] is False, (
        "this builder visits a set of strings, whose order depends on the process's hash seed, "
        "and your check certified it — did you regenerate in THIS process instead of in fresh "
        "processes with the hash seeds you were given?"
    )
    fd = bad["first_difference"]
    assert fd is not None and fd["line"] >= 1 and fd["expected"] != fd["got"], (
        "first_difference must name the 1-based line where the packs first differ, with both lines"
    )
    print("exercise 9 looks right")

In [ ]:
_try("exercise 9", _check_reproducibility)

## 12. The certificate

Now point the check at the real thing. The cell builds the first line's pack twice in this
process and compares, runs your check on it, and — so you can see what moved without
depending on this kernel's own hash seed — compares two fresh processes with each other,
field by field.

In [ ]:
def _show_first_line_finding() -> None:
    once = canonical_bytes(first_line_pack(SEED))
    twice = canonical_bytes(first_line_pack(SEED))
    print(f"first line's pack, built twice in this process: byte-identical = {once == twice}")
    cert = reproducibility_check(first_line_pack, SEED)
    print(f"your check, fresh processes with hash seeds {HASH_SEEDS}: "
          f"reproduced = {cert['reproduced']}")
    assert cert["reproduced"] is False, (
        "your check certified a pack whose figures depend on the hash seed"
    )
    a = json.loads(canonical_bytes(regenerate_in_fresh_process(first_line_pack, SEED,
                                                               HASH_SEEDS[0])))
    b = json.loads(canonical_bytes(regenerate_in_fresh_process(first_line_pack, SEED,
                                                               HASH_SEEDS[1])))
    moved = sorted(key for key in a if a[key] != b[key])
    print(f"\nfresh process with hash seed {HASH_SEEDS[0]} against hash seed {HASH_SEEDS[1]}: "
          f"{len(moved)} of {len(a)} fields differ")
    for key in moved:
        print(f"  {key:<40} {a[key]:.6f}  vs  {b[key]:.6f}")
    singles = [k for k in moved if k.startswith("importance.") and k.count(".") == 2]
    if moved and len(singles) == len(moved):
        print("\nEvery field that moved is a single-feature permutation importance. Nothing "
              "else did.")
    else:
        print("\nFields outside the single-feature importances moved as well: "
              + ", ".join(k for k in moved if k not in singles))


_try("the first line's pack", _show_first_line_finding,
     needs=tuple(_EXERCISES))

Every figure that moved came out of one call: the single-feature permutation importances.
They share one random generator, visited in the order of `columns` — and `first_line_pack`
builds that order from `set(COMMITTEE_ASKED)`. A set of strings iterates in an order that
depends on the process's hash seed. Each column gets a different slice of the generator's
stream in every process, so the numbers change, and in the same kernel they never do. The
seed was set. The pack still did not reproduce.

The fix is one line: de-duplicate in a way that keeps the committee's order. Then certify
again.

In [ ]:
def explanation_pack(seed: int) -> dict:
    """The corrected pack: the committee's order, de-duplicated, never a set's order."""
    asked = tuple(dict.fromkeys(COMMITTEE_ASKED))
    return assemble_pack(seed, asked)


def _show_certificate() -> None:
    global EXPLANATION_PACK, CERTIFICATE
    EXPLANATION_PACK = explanation_pack(SEED)
    CERTIFICATE = reproducibility_check(explanation_pack, SEED)
    assert CERTIFICATE["reproduced"] is True, (
        "the corrected pack should regenerate byte for byte — if it does not, something in "
        "your exercises depends on the process: a set, an unseeded generator, the clock"
    )
    text = canonical_bytes(EXPLANATION_PACK).decode()
    print(f"explanation pack: {len(EXPLANATION_PACK)} fields, "
          f"{len(text.splitlines())} lines of canonical JSON; the first few:")
    for line in text.splitlines()[1:9]:
        print("  " + line)
    print("\nREPRODUCIBILITY CERTIFICATE")
    print(json.dumps(CERTIFICATE, indent=1, sort_keys=True))


_try("the certificate", _show_certificate, needs=tuple(_EXERCISES))

## 13. Common mistakes

- **Ranking correlated features by single-feature permutation.** Each one stands in for the
  other, so each looks minor. Group them, and report the group.
- **Treating permutation as harmless.** Shuffling one of a correlated pair manufactures
  applicants who do not exist; section 6 counted them.
- **Reading partial dependence where the data is not.** The curve at the edge of the grid is
  the model's extrapolation, which two equally good models can disagree about completely.
- **Predicting at the average applicant.** Partial dependence and the baseline value are
  averages of predictions, not predictions at an average.
- **Taking efficiency as proof.** Attributions that add up can still be wrong in every
  coordinate; the sampled estimate did exactly that.
- **Not stating the value function.** A Shapley attribution without its baseline is a number
  without a unit. The pack records it.
- **Certifying in the same process.** A seed is necessary, not sufficient: anything that
  depends on the process — a set's order, the clock, an environment variable — survives it.
- **Serialising in insertion order.** It raises false alarms, and a check that cries wolf is
  switched off. The cell below shows it.

In [ ]:
def _names_by_set(seed: int) -> dict:
    """Identical CONTENT in every process, inserted in the order a set of strings is visited."""
    return {name: FEATURES.index(name) for name in set(FEATURES)}


def _show_false_alarm() -> None:
    naive = reproducibility_check(_names_by_set, SEED,
                                  serialise=lambda pack: json.dumps(pack).encode())
    canon = reproducibility_check(_names_by_set, SEED)
    print(f"same content, keys in insertion order:  reproduced = {naive['reproduced']}")
    print(f"same content, canonical bytes:          reproduced = {canon['reproduced']}")


_try("insertion order", _show_false_alarm, needs=("exercise 8", "exercise 9"))

## 14. Self-check

1. Shuffled alone, declared income and bureau income each cost the model little AUC;
   shuffled together they cost it more than any other feature. The committee asks whether
   income matters. The defensible answer is:
   - (a) no: each income feature is individually minor
   - (b) yes, and the evidence is the pair's grouped importance; each single figure is small
         because the other reading stands in for it
   - (c) the importances are inconsistent, so the model must be misspecified

2. At the top of the grid, nearly all of partial dependence's evaluation points for declared
   income lie outside the data's ellipse. The curve there describes:
   - (a) nothing at all: partial dependence is invalid for logistic models
   - (b) how default risk changes for applicants who declare a high income
   - (c) the model's behaviour on profiles that do not occur in the data, which the data
         cannot validate

3. Your Shapley attributions sum exactly to the prediction minus the baseline. This shows:
   - (a) only that efficiency holds — an estimate wrong in every coordinate can satisfy it
   - (b) that the attributions are correct
   - (c) that the baseline value function was the right one to choose

4. A pack built twice in one kernel is byte-identical; regenerated in three fresh processes,
   its single-feature importances differ. The most likely cause is:
   - (a) floating-point summation order inside numpy
   - (b) the fresh processes loaded a different numpy
   - (c) something iterates over a set of strings, whose order depends on the hash seed

5. A colleague's check compares packs after rounding every figure to four decimals, "to
   avoid spurious failures". What is wrong with it?
   - (a) a real non-determinism can hide below the fourth decimal, and the certificate would
         claim a reproduction that did not happen
   - (b) nothing, provided the report also rounds to four decimals
   - (c) it should round to six decimals instead

Answers are in this lesson's worked solution in the course repository.

In [ ]:
print(f"\nlesson wall time: {time.perf_counter() - _LESSON_T0:.1f}s")

## What you built, and where it goes next

An explanation pack with a certificate: importances with a spread and the grouping that
correlated features need, partial dependence with a measure of how much of it is
extrapolation, exact attributions under a stated value function, and a check that refuses to
certify what it cannot regenerate outside the process that made it. The validation-report
module of this programme takes the certificate's hash as the reference a committee pack cites,
so the explanation a reviewer reads and the explanation that was computed are provably the
same bytes.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_permutation_importance),
                              ("exercise 2", _check_grouped),
                              ("exercise 3", _check_partial_dependence),
                              ("exercise 4", _check_partial_dependence_2d),
                              ("exercise 5", _check_off_manifold),
                              ("exercise 6", _check_coalition_value),
                              ("exercise 7", _check_shapley),
                              ("exercise 8", _check_canonical),
                              ("exercise 9", _check_reproducibility)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))